# GastroVision — CNN training (v4: video-aware)

**Why v4:** v3 scored 94 % on held-out *still images* but failed on real endoscopy *videos* (HyperKvasir polyp videos were labelled `other` or the wrong disease). Training images are not video frames, so v4 adds the missing domain:

| Change | Why |
|---|---|
| **Frames from HyperKvasir `labeled-videos`** are added to training (1 frame/s, ≤ 40 per video) | the model finally sees video-style frames: on-screen text, overlays, octagonal mask, motion blur, different scopes |
| **Whole videos are held out** (all frames of a video stay in one split) | honest video-level test: detection rate, headline accuracy, false findings per video |
| **Your own clips are excluded from training** (`CFG.HOLDOUT_VIDEOS`) and checked at the end | independent proof it works on videos it never saw |
| More HyperKvasir *image* folders mapped to classes (`cecum`, `pylorus`, `z-line`, `esophagitis-a`, `…-b-d`, `ulcerative-colitis-grade-*`) | roughly doubles the data for the landmark / esophagitis / colitis classes |
| Overlay-box, timestamp-text and octagon-mask augmentation applied to **all** images | stops the network using overlays or the video look as a shortcut for a class |

**Known limit:** HyperKvasir labels a *whole video*, so some frames inside a `polyps` video show no polyp (label noise). The app's video verdict (consecutive-frame rule) is built for that.

**How to run (Kaggle)**
1. Settings → Accelerator **GPU T4**, Internet **On**.
2. *Add Data*: your Kvasir dataset, a HyperKvasir dataset **that contains `labeled-videos`** (if the data cell prints `video files found: 0`, upload `hyper-kvasir-labeled-videos.zip` from datasets.simula.no/hyper-kvasir as your own Kaggle dataset and attach it), and optionally Kvasir-SEG.
3. Run with `SMOKE = True` first (≈ 8 min), then `SMOKE = False` and **Save & Run All** (expect ≈ 1.5–2.5 h).
4. Download `gastrovision_model_bundle.zip` (≈ 75 MB) and the saved notebook with outputs.


In [ ]:
!pip -q install timm onnx onnxruntime onnxscript scikit-learn 2>&1 | tail -1

### 1 · Shared preprocessing module (identical file ships inside the desktop app)

In [ ]:
%%writefile gastrovision_preprocess.py
"""
gastrovision_preprocess.py
==========================
SINGLE SOURCE OF TRUTH for image preprocessing.

The exact same file is used by the training notebook and by the desktop app.
Most "great in the notebook, useless in the app" failures are caused by the two
sides preprocessing images slightly differently (colour order, normalisation,
resize method, borders).  Keeping one implementation removes that failure mode.

Colour convention: every public function in this module takes/returns **RGB**.
OpenCV (cv2.imread / VideoCapture) gives **BGR** - convert with `bgr_to_rgb`
before calling anything here.

Pipeline (identical for train-cache creation and for inference):
    rgb -> crop_black_borders -> remove_green_overlay -> limit_side
        -> resize to (img_size, img_size) [PIL bilinear] -> normalise -> CHW float32
"""
from __future__ import annotations

import json
import os
from typing import Dict, Tuple

import cv2
import numpy as np
from PIL import Image

DEFAULT_CFG: Dict = {
    "img_size": 384,
    "mean": [0.5, 0.5, 0.5],
    "std": [0.5, 0.5, 0.5],
    "cache_max_side": 512,
}


def bgr_to_rgb(frame_bgr: np.ndarray) -> np.ndarray:
    """OpenCV frames are BGR. The model was trained on RGB."""
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def black_border_box(rgb: np.ndarray, thr: int = 18, min_keep: float = 0.5):
    """Return (r0, r1, c0, c1) of the picture area inside any black frame."""
    h, w = rgb.shape[:2]
    if rgb.ndim != 3:
        return 0, h, 0, w
    bright = rgb.max(axis=2) > thr
    rows = np.where(bright.mean(axis=1) > 0.05)[0]
    cols = np.where(bright.mean(axis=0) > 0.05)[0]
    if len(rows) == 0 or len(cols) == 0:
        return 0, h, 0, w
    r0, r1, c0, c1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    if (r1 - r0) * (c1 - c0) < min_keep * h * w:
        return 0, h, 0, w
    return r0, r1, c0, c1


def crop_black_borders(rgb: np.ndarray, thr: int = 18, min_keep: float = 0.5) -> np.ndarray:
    """Trim the black frame that most endoscopy recorders add around the picture.
    Falls back to the untouched image if the crop would remove > (1-min_keep) of it."""
    r0, r1, c0, c1 = black_border_box(rgb, thr, min_keep)
    return np.ascontiguousarray(rgb[r0:r1, c0:c1])


def remove_green_overlay(rgb: np.ndarray, region: Tuple[float, float] = (0.45, 0.50),
                         min_frac: float = 0.01) -> np.ndarray:
    """Endoscopy recorders draw a solid green / teal 'scope position' inset in the bottom-left corner
    (bright green in Kvasir stills, teal in HyperKvasir videos).  It is not anatomy, and a network can learn it
    as a shortcut.  Detect it (hue 40-100 = green..teal, strongly saturated) and inpaint it away.

    Safety: the detected pixels must form a *solid box* (>= 50 % of their bounding box), so scattered
    greenish fluid is never inpainted.  No-op when absent, so ordinary frames are returned untouched."""
    h, w = rgb.shape[:2]
    r0, c1 = int(h * (1 - region[1])), int(w * region[0])
    patch = np.ascontiguousarray(rgb[r0:, :c1])
    if patch.size == 0:
        return rgb
    hsv = cv2.cvtColor(patch, cv2.COLOR_RGB2HSV)
    green = (hsv[..., 0] >= 40) & (hsv[..., 0] <= 100) & (hsv[..., 1] >= 110) & (hsv[..., 2] >= 90)
    if green.mean() < min_frac:
        return rgb
    ys, xs = np.where(green)
    box = green[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    if box.mean() < 0.5:
        return rgb
    pad_y, pad_x = max(3, int(0.02 * h)), max(3, int(0.02 * w))
    y0, y1 = max(0, ys.min() - pad_y), min(patch.shape[0], ys.max() + pad_y + 1)
    x0, x1 = max(0, xs.min() - pad_x), min(patch.shape[1], xs.max() + pad_x + 1)
    mask = np.zeros(patch.shape[:2], np.uint8)
    mask[y0:y1, x0:x1] = 255
    out = rgb.copy()
    out[r0:, :c1] = cv2.inpaint(patch, mask, 5, cv2.INPAINT_TELEA)
    return out


def clean_frame(rgb: np.ndarray) -> np.ndarray:
    """crop black borders + remove green overlay. RGB uint8 in, RGB uint8 out."""
    return remove_green_overlay(crop_black_borders(rgb))


def limit_side(img: Image.Image, max_side: int) -> Image.Image:
    w, h = img.size
    m = max(w, h)
    if m <= max_side:
        return img
    s = max_side / m
    return img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)


def pil_to_tensor_array(img: Image.Image, img_size: int, mean, std) -> np.ndarray:
    """PIL RGB -> float32 CHW, resized (squash) with PIL bilinear, normalised."""
    img = img.convert("RGB").resize((img_size, img_size), Image.BILINEAR)
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = (x - np.asarray(mean, np.float32)) / np.asarray(std, np.float32)
    return np.ascontiguousarray(x.transpose(2, 0, 1))


def preprocess_rgb(rgb: np.ndarray, cfg: Dict = None) -> np.ndarray:
    """Full inference-time preprocessing of ONE RGB frame -> float32 [3,S,S]."""
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    img = Image.fromarray(clean_frame(rgb))
    img = limit_side(img, int(cfg["cache_max_side"]))
    return pil_to_tensor_array(img, int(cfg["img_size"]), cfg["mean"], cfg["std"])


def preprocess_batch_bgr(frames_bgr, cfg: Dict = None) -> np.ndarray:
    """OpenCV (BGR) frames -> float32 [N,3,S,S].  The BGR->RGB swap lives HERE, once."""
    return np.stack([preprocess_rgb(bgr_to_rgb(f), cfg) for f in frames_bgr]).astype(np.float32)


def load_cfg(model_dir: str) -> Dict:
    p = os.path.join(model_dir, "preprocess.json")
    with open(p, "r", encoding="utf-8") as f:
        return {**DEFAULT_CFG, **json.load(f)}


In [ ]:
%%writefile gastrovision_video_scoring.py
"""
scoring/video_scoring.py
========================
Turns per-frame probabilities into ONE verdict for the whole video.

Why the old rule was unreliable
-------------------------------
The old code returned the class with the highest *average confidence*.  A single frame the
model was 99 % sure about (on a class seen once) beat a class seen in hundreds of frames.

New rule (all thresholds are parameters):
  1. smooth the per-class probability curves over time (median filter, 5 frames);
  2. a frame "votes" for a class only if it is the arg-max, is not the `other` class and is
     at least `conf_threshold` confident (calibrated);
  3. a class counts as DETECTED only if it has a run of >= `min_run` consecutive voting frames
     (a real lesion stays in view for a moment; single-frame noise does not);
  4. the headline verdict is taken from the most clinically important tier that has a detection:
         pathology  (polyps, esophagitis, ulcerative-colitis)
         > procedure (dyed-lifted-polyps, dyed-resection-margins)
         > landmark (normal-cecum, normal-pylorus, normal-z-line)
     and within a tier the class with most voting frames wins;
  5. nothing detected -> "no-finding".

The full list of findings (with timing) is always returned so the UI can show all of them.
"""
import json
import math
import os
from typing import Dict, List, Optional, Tuple

import numpy as np

PATHOLOGY = ("polyps", "esophagitis", "ulcerative-colitis")
PROCEDURE = ("dyed-lifted-polyps", "dyed-resection-margins")
LANDMARK = ("normal-cecum", "normal-pylorus", "normal-z-line")
TIER_ORDER = ["pathology", "procedure", "landmark"]
NO_FINDING = "no-finding"


def tier_of(cls: str) -> str:
    if cls in PATHOLOGY:
        return "pathology"
    if cls in PROCEDURE:
        return "procedure"
    if cls in LANDMARK:
        return "landmark"
    return "other"


def _median_smooth(P: np.ndarray, window: int = 5) -> np.ndarray:
    if window <= 1 or len(P) < 3:
        return P
    window = min(window, len(P) if len(P) % 2 == 1 else len(P) - 1)
    pad = window // 2
    Pp = np.pad(P, ((pad, pad), (0, 0)), mode="edge")
    win = np.lib.stride_tricks.sliding_window_view(Pp, window, axis=0)   # [n, K, window]
    return np.median(win, axis=-1)


def _longest_run(mask: np.ndarray) -> int:
    best = cur = 0
    for v in mask:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best


class video_scoring:
    # ---------------------------------------------------------------- loading
    @staticmethod
    def _video_dir(video_name: str) -> str:
        backend_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        return os.path.join(backend_dir, "outputs", video_name)

    def load_frame_scores(self, video_name: str) -> Tuple[List[str], List[str], np.ndarray]:
        names, classes, rows = [], None, []
        with open(os.path.join(self._video_dir(video_name), "frame_scores.jsonl"), "r", encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)
                if classes is None:
                    classes = list(d["probabilities"].keys())
                names.append(d["frame"])
                rows.append([d["probabilities"][c] for c in classes])
        return names, classes or [], np.asarray(rows, np.float32).reshape(len(names), len(classes or []))

    # ---------------------------------------------------------------- core
    def summarize_probs(self, names: List[str], classes: List[str], P: np.ndarray, conf_threshold: float = 0.7,
                        min_run: int = 3, smooth: int = 5, other_class: Optional[str] = "other") -> Dict:
        n = len(names)
        if n == 0:
            return {"label": NO_FINDING, "confidence": 0.0, "findings": [], "n_frames": 0, "n_rejected": 0,
                    "reason": "no usable frames"}
        Ps = _median_smooth(P, smooth)
        pred, conf = Ps.argmax(1), Ps.max(1)
        other_idx = classes.index(other_class) if other_class in classes else -1
        voting = (conf >= conf_threshold) & (pred != other_idx)
        findings = []
        for k, c in enumerate(classes):
            if k == other_idx:
                continue
            m = voting & (pred == k)
            run = _longest_run(m)
            if run < min_run:
                continue
            idx = np.where(m)[0]
            best = int(idx[np.argmax(P[idx, k])])
            findings.append({
                "class": c, "tier": tier_of(c), "frames": int(m.sum()), "longest_run": int(run),
                "mean_confidence": float(Ps[idx, k].mean()), "peak_confidence": float(P[best, k]),
                "first_frame": names[int(idx[0])], "best_frame": names[best], "best_index": best,
                "fraction_of_video": float(m.sum() / n),
            })
        tier_rank = {t: i for i, t in enumerate(TIER_ORDER + ["other"])}
        findings.sort(key=lambda f: (tier_rank[f["tier"]], -f["frames"], -f["mean_confidence"]))
        n_rejected = int((~voting).sum())
        if findings:
            top = findings[0]
            label, confidence = top["class"], top["mean_confidence"]
        else:
            # Nothing consistent was found.  `confidence` is 0 on purpose: it used to be the fraction of
            # rejected frames, which the UI showed as a misleading "100 % confidence".
            label, confidence = NO_FINDING, 0.0
        out = {"label": label, "confidence": float(confidence), "findings": findings, "n_frames": n,
               "n_rejected": n_rejected, "uncertain_fraction": float(n_rejected / n),
               "conf_threshold": conf_threshold, "min_run": min_run}
        if label == NO_FINDING:
            out["note"] = ("No target finding was seen consistently. This does NOT mean the video is normal: "
                           "the model only knows 8 findings plus 'other', and frames that look different from "
                           "its training images are rejected.")
        return out

    def summarize(self, video_name: str, conf_threshold: float = 0.7, min_run: int = 3, smooth: int = 5,
                  other_class: Optional[str] = "other") -> Dict:
        names, classes, P = self.load_frame_scores(video_name)
        out = self.summarize_probs(names, classes, P, conf_threshold, min_run, smooth, other_class)
        with open(os.path.join(self._video_dir(video_name), "video_summary.json"), "w", encoding="utf-8") as f:
            json.dump(out, f, indent=2)
        return out

    # ---------------------------------------------------------------- helpers
    def select_heatmap_frames(self, summary: Dict, video_name: str, top_n: int = 5, min_gap: int = 6) -> List[Dict]:
        """Pick up to `top_n` well-spread, high-confidence frames, round-robin over the detected findings."""
        names, classes, P = self.load_frame_scores(video_name)
        picked, per_class = [], []
        for f in summary.get("findings", []):
            k = classes.index(f["class"])
            order = [int(i) for i in np.argsort(-P[:, k]) if P[i, k] >= summary.get("conf_threshold", 0.7)]
            chosen = []
            for i in order:
                if all(abs(i - j) >= min_gap for j in chosen):
                    chosen.append(i)
            per_class.append((f["class"], chosen))
        r = 0
        while len(picked) < top_n and any(r < len(c) for _, c in per_class):
            for cls, c in per_class:
                if r < len(c) and len(picked) < top_n:
                    i = c[r]
                    k = classes.index(cls)
                    picked.append({"frame": names[i], "label": cls, "confidence": float(P[i, k]), "rank": len(picked) + 1})
            r += 1
        return picked

    # ---------------------------------------------------------------- backwards-compatible API
    def average_confidence(self, video_name: str, conf_threshold: float = 0.7):
        s = self.summarize(video_name, conf_threshold)
        return s["label"], s["confidence"]


### 2 · Config

In [ ]:
import os, sys, io, json, math, time, random, copy, shutil, warnings, datetime
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
import gastrovision_preprocess as gp

class CFG:
    SMOKE = False                      # <<< True = 5-minute dry run. Set False for the real run.
    SEED = 42
    TARGET_CLASSES = ["dyed-lifted-polyps", "dyed-resection-margins", "esophagitis", "normal-cecum",
                      "normal-pylorus", "normal-z-line", "polyps", "ulcerative-colitis"]
    OTHER_NAME = "other"
    # HyperKvasir folders that are NOT in the 8 target classes -> become the `other` class
    OTHER_FOLDERS = ["barretts", "barretts-short-segment", "bbps-0-1", "bbps-2-3", "hemorrhoids",
                     "ileum", "impacted-stool", "retroflex-rectum", "retroflex-stomach"]
    EXTRA_OTHER_DIRS = []             # any extra folders of "not one of the 8" frames, e.g. a sample of video frames
    OTHER_MAX = 1200                  # cap on `other` images (keeps it roughly class-sized)
    # ---- video-domain training data (HyperKvasir *labeled-videos*) ------------------------------------------------
    USE_VIDEOS = True
    VIDEO_FPS = 1.0                   # frames per second sampled from every labelled video
    VIDEO_MAX_FRAMES = 40             # cap per video (spread evenly over the video)
    VIDEO_OTHER_MAX_FRAMES = 20       # cap per video for videos that map to the `other` class
    # Videos listed here (file name without extension) are NEVER used for train/val/test; they are checked separately at the end.
    HOLDOUT_VIDEOS = ["16e7e461-a9a7-4b04-8d2c-155de3eb1c90", "29ce8d04-c184-42fa-ac05-322d3a2d37bd",
                      "2a200509-e1d3-46bd-8ae7-7fb1a8491709"]
    BACKBONE = "tf_efficientnetv2_s.in21k_ft_in1k"   # alternatives: "efficientnet_b3.ra2_in1k", "resnet50.a1_in1k", "densenet121.ra_in1k"
    IMG_SIZE = 384
    CACHE_MAX_SIDE = 512
    BATCH = 16                        # must be even (MixUp). Lower to 16 if CUDA OOM
    EPOCHS = 30
    PATIENCE = 7
    LR = 2e-4                         # backbone; the new head uses 5x
    WD = 0.02
    WARMUP_EPOCHS = 1
    EMA_DECAY = 0.998
    LABEL_SMOOTH = 0.1
    MIX_PROB = 0.5
    DROP_RATE, DROP_PATH = 0.3, 0.2
    WORKERS = 4
    SIM_START = 0.85                  # embedding-similarity grouping (adaptive, see below)
    MAX_GROUP_FRAC = 0.02
    KVASIR_SEG = None                 # auto-detected if a Kvasir-SEG dataset is attached
    WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "./work"

if CFG.SMOKE:
    CFG.IMG_SIZE, CFG.BATCH, CFG.EPOCHS, CFG.PATIENCE, CFG.OTHER_MAX = 224, 8, 1, 1, 24
    CFG.VIDEO_MAX_FRAMES = CFG.VIDEO_OTHER_MAX_FRAMES = 6
os.makedirs(CFG.WORK, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG.SEED)
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU - this will be slow", "| smoke:", CFG.SMOKE)
if DEVICE != "cuda" and not CFG.SMOKE:
    raise SystemExit("Enable a GPU accelerator in Kaggle settings.")

### 3 · Find the data (merges any `train/`, `val/` folders — we re-split properly below)

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
VID_EXT = {".avi", ".mp4", ".mov", ".mkv", ".mpg", ".mpeg", ".wmv", ".m4v"}
EXTRA_MAP = {"cecum": "normal-cecum", "pylorus": "normal-pylorus", "z-line": "normal-z-line"}   # HyperKvasir names -> Kvasir classes
_T = {c.lower() for c in CFG.TARGET_CLASSES}; _O = {f.lower() for f in CFG.OTHER_FOLDERS}
def map_folder(name):
    """folder name -> training class (or None if it is not used)."""
    n = name.lower()
    if n in _T: return n
    if n in EXTRA_MAP: return EXTRA_MAP[n]
    if n.startswith("esophagitis"): return "esophagitis"                       # esophagitis-a, esophagitis-b-d
    if n.startswith("ulcerative-colitis"): return None if n.endswith("grade-0-1") else "ulcerative-colitis"   # grade 0-1 is ambiguous -> skipped
    if n in _O: return CFG.OTHER_NAME
    return None

def discover(root="/kaggle/input"):
    found = defaultdict(list)
    for dp, _, fns in os.walk(root):
        if any(Path(f).suffix.lower() in IMG_EXT for f in fns):
            found[os.path.basename(dp).lower()].append(dp)
    return found
def discover_videos(root="/kaggle/input"):
    out = []
    for dp, _, fns in os.walk(root):
        for f in fns:
            if Path(f).suffix.lower() in VID_EXT:
                cls = next((map_folder(part) for part in reversed(Path(dp).parts) if map_folder(part)), None)
                out.append((str(Path(dp) / f), cls))
    return out
def list_imgs(d):
    return sorted(str(Path(d) / f) for f in os.listdir(d) if Path(f).suffix.lower() in IMG_EXT)

found = discover()
rng = random.Random(CFG.SEED)
rows = []
for c in CFG.TARGET_CLASSES:
    dirs = found.get(c.lower(), [])
    paths = sorted({p for d in dirs for p in list_imgs(d)})
    print(f"{c:24s} {len(paths):5d} images from {len(dirs)} folder(s): {[d.split('input/')[-1] for d in dirs][:3]}")
    if not paths:
        raise SystemExit(f"No folder named '{c}' found under /kaggle/input - attach the dataset.")
    if CFG.SMOKE: paths = rng.sample(paths, min(12, len(paths)))
    rows += [(p, c) for p in paths]
# extra HyperKvasir *image* folders that belong to a target class (cecum, pylorus, z-line, esophagitis-a, ulcerative-colitis-grade-*)
for name, dirs in sorted(found.items()):
    m = map_folder(name)
    if m in _T and name.lower() != m:
        ps = sorted({p for d in dirs for p in list_imgs(d)})
        if CFG.SMOKE: ps = rng.sample(ps, min(6, len(ps)))
        print(f"  {m:24s} <- extra image folder '{name}'  {len(ps)}"); rows += [(p, m) for p in ps]

other_pool = []
for f in CFG.OTHER_FOLDERS:
    ps = sorted({p for d in found.get(f.lower(), []) for p in list_imgs(d)})
    if ps: print(f"  other <- {f:24s} {len(ps)}"); other_pool.append(ps)
for d in CFG.EXTRA_OTHER_DIRS:
    ps = list_imgs(d); print(f"  other <- {d} {len(ps)}"); other_pool.append(ps)
HAS_OTHER = len(other_pool) > 0
if HAS_OTHER:
    per = max(1, CFG.OTHER_MAX // len(other_pool))
    for ps in other_pool:
        rows += [(p, CFG.OTHER_NAME) for p in rng.sample(ps, min(per, len(ps)))]

# ---------------- video frames (the missing piece: training images are NOT video frames) ----------------
import cv2, subprocess
vrows = []
if CFG.USE_VIDEOS:
    allv = discover_videos()
    vids = [(p, c) for p, c in allv if c is not None and "unlabeled" not in p.lower()]
    vids = list({Path(p).stem: (p, c) for p, c in vids}.values())                    # one entry per video id
    print(f"\nvideo files found: {len(allv)} | usable (folder maps to a class): {len(vids)}")
    if CFG.SMOKE:
        by = defaultdict(list)
        for v in vids: by[v[1]].append(v)
        keep_v = [x for c in by for x in rng.sample(by[c], min(2, len(by[c])))] + [v for v in vids if Path(v[0]).stem in CFG.HOLDOUT_VIDEOS]
        vids = list({v[0]: v for v in keep_v}.values())
    if not vids:
        print("!! No labelled videos found. The model will again be trained on still images only and will NOT be robust on video.\n"
              "   Fix: download hyper-kvasir-labeled-videos.zip from datasets.simula.no/hyper-kvasir, upload it to Kaggle as a Dataset, attach it, re-run.")
    else:
        VF = Path(CFG.WORK) / "video_frames"; shutil.rmtree(VF, ignore_errors=True); VF.mkdir(parents=True)
        def extract_video(a):
            i, (path, cls) = a
            stem = Path(path).stem
            cap_n = 200 if stem in CFG.HOLDOUT_VIDEOS else (CFG.VIDEO_OTHER_MAX_FRAMES if cls == CFG.OTHER_NAME else CFG.VIDEO_MAX_FRAMES)
            out_dir = VF / f"{i:04d}"; out_dir.mkdir(parents=True, exist_ok=True); saved = []
            try:
                cap = cv2.VideoCapture(path)
                fps = cap.get(cv2.CAP_PROP_FPS); fps = fps if fps and 1 < fps < 240 else 25.0
                total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
                step = max(1, int(round(fps / CFG.VIDEO_FPS)))
                if total > 0:
                    idxs = list(range(step // 2, total, step))
                    if len(idxs) > cap_n: idxs = [idxs[int(k)] for k in np.linspace(0, len(idxs) - 1, cap_n)]
                    want, last, k = set(idxs), (max(idxs) if idxs else -1), 0
                    while k <= last:
                        if not cap.grab(): break
                        if k in want:
                            ok, fr = cap.retrieve()
                            if ok:
                                fp = out_dir / f"f{k:06d}.jpg"; cv2.imwrite(str(fp), fr, [cv2.IMWRITE_JPEG_QUALITY, 95]); saved.append(str(fp))
                        k += 1
                cap.release()
            except Exception as e:
                print("  cv2 failed on", stem, repr(e))
            if not saved:          # fall back to the ffmpeg binary
                try:
                    subprocess.run(["ffmpeg", "-v", "error", "-i", path, "-vf", f"fps={CFG.VIDEO_FPS}", "-frames:v", str(cap_n),
                                    "-q:v", "2", str(out_dir / "f%06d.jpg")], check=False, timeout=300)
                    saved = sorted(str(p) for p in out_dir.glob("*.jpg"))
                except Exception as e:
                    print("  ffmpeg failed on", stem, repr(e))
            return [(p, cls, path) for p in saved]
        t0 = time.time()
        with ThreadPoolExecutor(4) as ex:
            for r in ex.map(extract_video, list(enumerate(vids))): vrows += r
        nvid = Counter(c for _, c in vids); nfr = Counter(c for _, c, _ in vrows)
        print(f"extracted {len(vrows)} frames from {len(vids)} videos in {time.time()-t0:.0f}s")
        for c in CFG.TARGET_CLASSES + [CFG.OTHER_NAME]:
            if nvid.get(c): print(f"  video class {c:24s} {nvid[c]:4d} videos {nfr.get(c, 0):6d} frames")
        failed = [Path(p).stem for p, _ in vids if not any(v == p for _, _, v in vrows)]
        if failed: print("  videos with 0 readable frames:", failed[:10])
        if any(c == CFG.OTHER_NAME for _, c, _ in vrows): HAS_OTHER = True
if not HAS_OTHER:
    print("\n!! No 'other' source found. Training 8 classes only.")

CLASSES = CFG.TARGET_CLASSES + ([CFG.OTHER_NAME] if HAS_OTHER else [])
K = len(CLASSES); C2I = {c: i for i, c in enumerate(CLASSES)}
df = pd.DataFrame(rows, columns=["path", "cls"]).drop_duplicates("path").reset_index(drop=True)
df["vid"] = None
if vrows:
    df = pd.concat([df, pd.DataFrame(vrows, columns=["path", "cls", "vid"])], ignore_index=True)
df = df[df.cls.isin(CLASSES)].reset_index(drop=True)
df["holdout"] = df.vid.map(lambda v: isinstance(v, str) and Path(v).stem in CFG.HOLDOUT_VIDEOS)
df["y"] = df.cls.map(C2I)
df["domain"] = np.where(df.vid.notna(), "video", "still")
print("\nclasses:", CLASSES); print(pd.crosstab(df.cls, df.domain).to_string())


### 4 · Clean + cache images (same `clean_frame` the app uses) and hash them

In [ ]:
CACHE = Path(CFG.WORK) / "cache"; shutil.rmtree(CACHE, ignore_errors=True); CACHE.mkdir(parents=True)
def dhash64(img):
    g = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return np.packbits((g[:, 1:] > g[:, :-1]).flatten())
def cache_one(a):
    i, path = a
    try:
        rgb = np.asarray(Image.open(path).convert("RGB"))
        img = gp.limit_side(Image.fromarray(gp.clean_frame(rgb)), CFG.CACHE_MAX_SIDE)
        out = CACHE / f"{i:05d}.png"; img.save(out, compress_level=1)
        return i, str(out), dhash64(img), None
    except Exception as e:
        return i, None, None, repr(e)
t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    res = list(ex.map(cache_one, list(enumerate(df.path))))
bad = [r for r in res if r[1] is None]
print(f"cached {len(res)-len(bad)} images in {time.time()-t0:.0f}s; unreadable: {len(bad)}")
for r in bad[:5]: print("  ", df.path[r[0]], r[3])
df["cache"] = [r[1] for r in res]; hashes = [r[2] for r in res]
keep = df.cache.notna().values
df = df[keep].reset_index(drop=True); hashes = np.stack([h for h, k in zip(hashes, keep) if k])
hold = df.holdout.values
EXT = df[hold].reset_index(drop=True); df = df[~hold].reset_index(drop=True); hashes = hashes[~hold]
print(f"held out {len(EXT)} frames from {EXT.vid.nunique()} external video(s) (never trained/validated on): {sorted(Path(v).stem[:8] for v in EXT.vid.unique())}")


### 5 · Group near-duplicates → leakage-free split
Frames from the same video/patient look almost identical. If one lands in train and its twin in val, accuracy is inflated.
Images are linked when (a) their perceptual hash is almost identical **or** (b) their CNN embeddings are very similar;
linked images form a *group*, and whole groups go to exactly one split. The similarity threshold adapts so that no
group exceeds 2 % of the data.

In [ ]:
class DSU:
    def __init__(s, n): s.p = list(range(n))
    def f(s, x):
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def u(s, a, b): s.p[s.f(a)] = s.f(b)

n = len(df)
VID_GROUPS = [list(ix) for ix in df.groupby("vid").indices.values()] if df.vid.notna().any() else []
print("videos kept as indivisible groups:", len(VID_GROUPS))
# (a) perceptual-hash near-duplicates (Hamming <= 4 of 64 bits)
bits = np.unpackbits(hashes, axis=1).astype(np.float32)
hash_pairs = []
for s in range(0, n, 1000):
    A = bits[s:s+1000]
    ham = 64 - (A @ bits.T + (1 - A) @ (1 - bits).T)
    ii, jj = np.where(ham <= 4)
    hash_pairs += [(a + s, b) for a, b in zip(ii, jj) if a + s < b]
print("hash-duplicate pairs:", len(hash_pairs))

# (b) embedding neighbours from the *pre-trained, not fine-tuned* backbone
class EmbDS(torch.utils.data.Dataset):
    def __init__(s, paths, size, mean, std): s.p, s.sz, s.m, s.s = list(paths), size, mean, std
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return torch.from_numpy(gp.pil_to_tensor_array(Image.open(s.p[i]), s.sz, s.m, s.s))
emb_model = timm.create_model(CFG.BACKBONE, pretrained=True, num_classes=0).to(DEVICE).eval()
dc = timm.data.resolve_data_config({}, model=emb_model); MEAN, STD = list(dc["mean"]), list(dc["std"])
print("normalisation from model:", MEAN, STD)
feats = []
with torch.no_grad():
    for x in torch.utils.data.DataLoader(EmbDS(df.cache, 256, MEAN, STD), batch_size=64, num_workers=CFG.WORKERS):
        feats.append(emb_model(x.to(DEVICE)).float().cpu())
feats = torch.cat(feats); feats = feats - feats.mean(0, keepdim=True); feats = F.normalize(feats, dim=1)
del emb_model; torch.cuda.empty_cache()
kk = min(8, n - 1); nb_sim, nb_idx = [], []
for s in range(0, n, 1024):
    sim = feats[s:s+1024] @ feats.T
    for r in range(sim.shape[0]): sim[r, s + r] = -1
    v, ix = sim.topk(kk, dim=1); nb_sim.append(v); nb_idx.append(ix)
nb_sim, nb_idx = torch.cat(nb_sim).numpy(), torch.cat(nb_idx).numpy()

# ---- group near-duplicates WITHOUT chaining everything into one blob --------------------------------------------------
# v4 bug: single-linkage over loose 64-bit hashes (26 000 "twin" pairs) chained 8 878 frames into ONE group, which then
# swallowed most of the data and left val/test almost empty.  Now: (1) a hash twin only counts if the CNN embedding agrees,
# (2) strongest links are merged first, (3) a group may never exceed MAX_GROUP frames.
SIM_THR = 0.95
MAX_GROUP = max(60, int(0.006 * n))
thr = SIM_THR
Fn = np.asarray(feats, dtype=np.float32)
edges = {}
ii, jj = np.where(nb_sim >= SIM_THR)
for a_, k_ in zip(ii, jj):
    a_ = int(a_); b_ = int(nb_idx[a_, k_]); key = (a_, b_) if a_ < b_ else (b_, a_)
    edges[key] = max(edges.get(key, 0.0), float(nb_sim[a_, k_]))
if hash_pairs:
    hp = np.array(hash_pairs); hs = (Fn[hp[:, 0]] * Fn[hp[:, 1]]).sum(1); n_ok = 0
    for (a_, b_), s_ in zip(hash_pairs, hs):
        if s_ >= SIM_THR: edges[(a_, b_)] = max(edges.get((a_, b_), 0.0), float(s_)); n_ok += 1
    print(f"hash twins confirmed by the CNN: {n_ok} of {len(hash_pairs)}")
class CappedDSU:
    def __init__(s, n, cap): s.p = list(range(n)); s.sz = [1] * n; s.cap = cap
    def f(s, x):
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def u(s, a, b):
        ra, rb = s.f(a), s.f(b)
        if ra == rb: return True
        if s.sz[ra] + s.sz[rb] > s.cap: return False
        s.p[ra] = rb; s.sz[rb] += s.sz[ra]; return True
dsu = CappedDSU(n, MAX_GROUP)
for ix in VID_GROUPS:
    for j in ix[1:]: dsu.u(ix[0], j)
refused = 0
for (a_, b_), s_ in sorted(edges.items(), key=lambda kv: -kv[1]):
    if not dsu.u(a_, b_): refused += 1
roots = np.array([dsu.f(i) for i in range(n)]); cnt = Counter(roots)
sizes = np.array(sorted(cnt.values(), reverse=True))
print(f"similarity threshold {SIM_THR} | groups: {len(sizes)} for {n} frames | largest group: {sizes[0]} (cap {MAX_GROUP}) | "
      f"frames with a near-twin: {int(sizes[sizes > 1].sum())} ({100 * sizes[sizes > 1].sum() / n:.1f}%) | links refused by the cap: {refused}")
assert sizes[0] <= max(MAX_GROUP, max((len(v) for v in VID_GROUPS), default=0)), "group cap violated"

# drop exact duplicates (the same frame present in Kvasir AND HyperKvasir) - keeps one copy per group, speeds training ~25 %
keep = np.ones(n, bool); by_g = defaultdict(list)
for i, r in enumerate(roots): by_g[r].append(i)
for r, ix in by_g.items():
    if len(ix) < 2: continue
    Sm = Fn[ix] @ Fn[ix].T; kept = []
    for t_ in range(len(ix)):
        if any(Sm[t_, kt] >= 0.995 for kt in kept): keep[ix[t_]] = False
        else: kept.append(t_)
print(f"exact duplicates dropped: {int((~keep).sum())} -> {int(keep.sum())} frames remain")
df = df[keep].reset_index(drop=True); roots = roots[keep]
df["group"] = pd.factorize(roots)[0]


### 6 · Train / val / test split (≈ 67 / 17 / 17, stratified, group-aware)

In [ ]:
y = df.y.values; g = df.group.values
outer = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=CFG.SEED)
rest_idx, test_idx = next(outer.split(df, y, g))
inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.SEED)
tr_i, va_i = next(inner.split(df.iloc[rest_idx], y[rest_idx], g[rest_idx]))
df["split"] = ""
df.loc[rest_idx[tr_i], "split"] = "train"; df.loc[rest_idx[va_i], "split"] = "val"; df.loc[test_idx, "split"] = "test"
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not (set(df[df.split == a].group) & set(df[df.split == b].group)), f"group leakage {a}/{b}"
print("no group appears in two splits  OK"); print(pd.crosstab(df.cls, df.split).to_string())
df.drop(columns=["cache"]).to_csv(f"{CFG.WORK}/splits.csv", index=False)
vt = df[df.vid.notna()].drop_duplicates("vid")
if len(vt): print("\nvideos per split (whole videos are held out):"); print(pd.crosstab(vt.cls, vt.split).to_string())
need = 1 if CFG.SMOKE else 25
for sp_ in ("val", "test"):
    cc = df[df.split == sp_].cls.value_counts(); miss = [c for c in CLASSES if cc.get(c, 0) < need]
    if miss and CFG.SMOKE: print(f"(smoke dry run: only 264 frames, so the {sp_} split may lack {miss} - ignored; the real run enforces this)")
    if miss and not CFG.SMOKE: raise RuntimeError(f"STOP - the {sp_} split has fewer than {need} frames for {miss}: grouping is too coarse. Do not train.")
print(f"split sanity gate passed (every class has >= {need} frames in val and test)")


### 7 · Datasets & augmentation (simulating real video: blur, low resolution, compression)

In [ ]:
from torchvision.transforms import v2 as T
IMG = CFG.IMG_SIZE
from PIL import ImageDraw, ImageFont
class RandOverlayBox:      # coloured inset boxes (scope-position insets, recorder overlays) on ANY class
    COL = [(20, 160, 140), (0, 200, 0), (30, 30, 30), (200, 200, 200), (0, 90, 200), (210, 60, 60)]
    def __init__(s, p=0.35): s.p = p
    def __call__(s, im):
        if random.random() < s.p:
            im = im.copy(); w, h = im.size; bw, bh = int(w * random.uniform(.12, .3)), int(h * random.uniform(.12, .3))
            x0 = 0 if random.random() < .6 else random.randint(0, max(0, w - bw)); y0 = h - bh if random.random() < .6 else random.randint(0, max(0, h - bh))
            ImageDraw.Draw(im).rectangle([x0, y0, x0 + bw, y0 + bh], fill=random.choice(s.COL))
        return im
class RandTimestamp:       # on-screen text (date/time/recorder status) like HyperKvasir videos
    def __init__(s, p=0.3): s.p = p
    def __call__(s, im):
        if random.random() < s.p:
            im = im.copy(); w, h = im.size; d = ImageDraw.Draw(im)
            try: f = ImageFont.load_default(size=random.randint(10, 22))
            except Exception: f = ImageFont.load_default()
            for _ in range(random.randint(1, 4)):
                t = random.choice([f"{random.randint(1,28):02d}/{random.randint(1,12):02d}/20{random.randint(8,20)}",
                                   f"{random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}",
                                   f"CVP:{random.randint(5,30)}", "Video 576i", f"{random.random()*9:.2f} Mbit/s", "0 Buf", "Q:N"])
                d.text((random.randint(0, int(w * .5)), random.randint(0, int(h * .85))), t, fill=random.choice([(255, 255, 255), (255, 255, 0), (200, 200, 255)]), font=f)
        return im
class RandOctagon:         # black corners of the endoscope's octagonal picture mask
    def __init__(s, p=0.3): s.p = p
    def __call__(s, im):
        if random.random() < s.p:
            im = im.copy(); w, h = im.size; c = int(min(w, h) * random.uniform(.08, .22)); d = ImageDraw.Draw(im)
            for pts in ([(0, 0), (c, 0), (0, c)], [(w, 0), (w - c, 0), (w, c)], [(0, h), (c, h), (0, h - c)], [(w, h), (w - c, h), (w, h - c)]):
                d.polygon(pts, fill=(0, 0, 0))
        return im
class RandRot90:
    def __call__(s, im):
        k = random.randint(0, 3); return im.rotate(90 * k, expand=True) if k else im
class RandDownUp:
    def __init__(s, p=0.3, lo=0.35, hi=0.8): s.p, s.lo, s.hi = p, lo, hi
    def __call__(s, im):
        if random.random() < s.p:
            w, h = im.size; f = random.uniform(s.lo, s.hi)
            im = im.resize((max(8, int(w*f)), max(8, int(h*f))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        return im
class RandJPEG:
    def __init__(s, p=0.3, q=(35, 90)): s.p, s.q = p, q
    def __call__(s, im):
        if random.random() < s.p:
            b = io.BytesIO(); im.save(b, "JPEG", quality=random.randint(*s.q)); b.seek(0); im = Image.open(b).convert("RGB")
        return im
class RandCornerPatch:   # residual-overlay shortcut breaker
    def __init__(s, p=0.3): s.p = p
    def __call__(s, x):
        if random.random() < s.p:
            _, h, w = x.shape; x[:, h - int(h*random.uniform(.15, .35)):, :int(w*random.uniform(.15, .35))] = 0
        return x
train_tf = T.Compose([
    RandOverlayBox(0.35), RandTimestamp(0.3), RandOctagon(0.3),
    RandRot90(),
    T.RandomResizedCrop(IMG, scale=(0.45, 1.0), ratio=(0.85, 1.2), antialias=True),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.ColorJitter(0.25, 0.25, 0.2, 0.02),
    RandDownUp(0.3), RandJPEG(0.3),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.8))], p=0.2),
    T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.1), value=0),
    RandCornerPatch(0.3),
])
def degrade_pil(im):   # fixed, harsh "bad video frame" used only for the robustness test
    w, h = im.size
    im = im.resize((w // 2, h // 2), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    from PIL import ImageFilter
    im = im.filter(ImageFilter.GaussianBlur(1.2))
    b = io.BytesIO(); im.save(b, "JPEG", quality=40); b.seek(0); return Image.open(b).convert("RGB")

class GIDataset(torch.utils.data.Dataset):
    def __init__(s, d, train, degrade=False):
        s.p, s.y, s.train, s.deg = d.cache.tolist(), d.y.tolist(), train, degrade
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        im = Image.open(s.p[i]).convert("RGB")
        if s.train: x = train_tf(im)
        else:
            if s.deg: im = degrade_pil(im)
            x = torch.from_numpy(gp.pil_to_tensor_array(im, IMG, MEAN, STD))
        return x, s.y[i]
D = {k: df[df.split == k].reset_index(drop=True) for k in ["train", "val", "test"]}
cw = 1.0 / D["train"].y.value_counts().sort_index().reindex(range(K), fill_value=1).values
sampler = torch.utils.data.WeightedRandomSampler(cw[D["train"].y.values], num_samples=len(D["train"]), replacement=True)
mk = lambda d, **kw: torch.utils.data.DataLoader(GIDataset(d, **kw), batch_size=CFG.BATCH, num_workers=CFG.WORKERS, pin_memory=True,
                                                  persistent_workers=CFG.WORKERS > 0)
train_loader = torch.utils.data.DataLoader(GIDataset(D["train"], True), batch_size=CFG.BATCH, sampler=sampler, drop_last=True,
                                           num_workers=CFG.WORKERS, pin_memory=True, persistent_workers=CFG.WORKERS > 0)
val_loader, test_loader = mk(D["val"], train=False), mk(D["test"], train=False)
test_deg_loader = mk(D["test"], train=False, degrade=True)
print({k: len(v) for k, v in D.items()}, "| steps/epoch:", len(train_loader))

### 8 · Model, EMA, training loop

In [ ]:
def build_model(pretrained=True):
    try:
        return timm.create_model(CFG.BACKBONE, pretrained=pretrained, num_classes=K,
                                 drop_rate=CFG.DROP_RATE, drop_path_rate=CFG.DROP_PATH)
    except Exception as e:
        raise SystemExit(f"Could not create/download {CFG.BACKBONE}: {e}\nIs Internet enabled in the Kaggle notebook settings?")

class EMA:
    def __init__(s, m, decay):
        s.m = copy.deepcopy(m).eval(); s.d = decay
        for p in s.m.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(s, model):
        msd = model.state_dict()
        for k, v in s.m.state_dict().items():
            if v.dtype.is_floating_point: v.mul_(s.d).add_(msd[k].detach(), alpha=1 - s.d)
            else: v.copy_(msd[k])

@torch.no_grad()
def predict_logits(model, loader, tta=False):
    model.eval(); out = []
    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            lo = model(x)
            if tta: lo = (lo + model(x.flip(3)) + model(x.flip(2))) / 3
        out.append(lo.float().cpu())
    return torch.cat(out)

def param_groups(model):
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    g = {(h, d): [] for h in (0, 1) for d in (0, 1)}
    for p in model.parameters():
        if p.requires_grad: g[(int(id(p) in head_ids), int(p.ndim > 1))].append(p)
    return [dict(params=ps, lr=CFG.LR * (5 if h else 1), weight_decay=CFG.WD if d else 0.0) for (h, d), ps in g.items() if ps]

def train():
    seed_all(CFG.SEED)
    model = build_model().to(DEVICE).to(memory_format=torch.channels_last)
    ema = EMA(model, CFG.EMA_DECAY)
    opt = torch.optim.AdamW(param_groups(model))
    base = [g["lr"] for g in opt.param_groups]
    spe = len(train_loader); total, warm = CFG.EPOCHS * spe, CFG.WARMUP_EPOCHS * spe
    def lr_at(s):
        return s / max(1, warm) if s < warm else max(0.01, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    mix = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=CFG.MIX_PROB, switch_prob=0.5, mode="batch",
                label_smoothing=CFG.LABEL_SMOOTH, num_classes=K)
    crit = SoftTargetCrossEntropy()
    best, bad, hist, step = -1, 0, [], 0
    for ep in range(CFG.EPOCHS):
        model.train(); t0 = time.time(); ls = 0
        for x, yb in train_loader:
            for gi, g in enumerate(opt.param_groups): g["lr"] = base[gi] * lr_at(step)
            x = x.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last); yb = yb.to(DEVICE)
            x, ys = mix(x, yb)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                loss = crit(model(x), ys)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
            ema.update(model); step += 1; ls += loss.item()
        lo = predict_logits(ema.m, val_loader); yv = torch.tensor(D["val"].y.values)
        acc = (lo.argmax(1) == yv).float().mean().item(); f1 = f1_score(yv, lo.argmax(1), average="macro")
        vl = F.cross_entropy(lo, yv).item()
        hist.append(dict(epoch=ep + 1, train_loss=ls / spe, val_loss=vl, val_acc=acc, val_f1=f1))
        flag = ""
        if f1 > best + 1e-4:
            best, bad = f1, 0; torch.save(ema.m.state_dict(), f"{CFG.WORK}/best_ema.pt"); flag = "  <- best"
        else: bad += 1
        print(f"ep {ep+1:2d}/{CFG.EPOCHS} | {time.time()-t0:4.0f}s | train {ls/spe:.3f} | val loss {vl:.3f} acc {acc:.4f} macroF1 {f1:.4f}{flag}")
        if bad >= CFG.PATIENCE: print("early stop"); break
    return pd.DataFrame(hist)
hist = train()

### 9 · Honest evaluation
Everything below uses the **test split**, which was never used for training, checkpoint choice, calibration or thresholds.

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
model = build_model(pretrained=False); model.load_state_dict(torch.load(f"{CFG.WORK}/best_ema.pt", map_location="cpu"))
model = model.to(DEVICE).to(memory_format=torch.channels_last).eval()
yv, yt = torch.tensor(D["val"].y.values), torch.tensor(D["test"].y.values)
lv, lt = predict_logits(model, val_loader), predict_logits(model, test_loader)

# temperature scaling fitted on VAL only
logT = torch.zeros(1, requires_grad=True); o = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
def _c():
    o.zero_grad(); l = F.cross_entropy(lv / logT.exp(), yv); l.backward(); return l
o.step(_c); TEMP = float(logT.exp().detach())

def ece(p, y, bins=15):
    conf, pred = p.max(1); acc = (pred == y).float(); e = 0.
    for lo_, hi_ in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo_) & (conf <= hi_)
        if m.any(): e += m.float().mean().item() * abs(acc[m].mean().item() - conf[m].mean().item())
    return e
pv, pt = F.softmax(lv / TEMP, 1), F.softmax(lt / TEMP, 1)
print(f"temperature T = {TEMP:.3f} | test ECE before {ece(F.softmax(lt,1), yt):.4f} -> after {ece(pt, yt):.4f}")

pred = lt.argmax(1)
acc, f1m = accuracy_score(yt, pred), f1_score(yt, pred, average="macro", labels=list(range(K)), zero_division=0)
lt_tta = predict_logits(model, test_loader, tta=True); acc_tta = accuracy_score(yt, lt_tta.argmax(1))
lt_deg = predict_logits(model, test_deg_loader); acc_deg = accuracy_score(yt, lt_deg.argmax(1))
print(f"\nTEST accuracy {acc:.4f} | macro-F1 {f1m:.4f} | with flip-TTA {acc_tta:.4f} | on degraded (blur+low-res+JPEG) frames {acc_deg:.4f}")
rep = classification_report(yt, pred, labels=list(range(K)), target_names=CLASSES, output_dict=True, digits=4, zero_division=0)
print(classification_report(yt, pred, labels=list(range(K)), target_names=CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(yt, pred, labels=range(K))
fig, ax = plt.subplots(figsize=(8, 7)); ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.set_xticklabels(CLASSES, rotation=60, ha="right"); ax.set_yticklabels(CLASSES)
for i in range(K):
    for j in range(K): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Test confusion matrix"); plt.tight_layout()
plt.savefig(f"{CFG.WORK}/confusion_matrix.png", dpi=130); plt.show()
if len(hist) > 1:
    hist.plot(x="epoch", y=["train_loss", "val_loss", "val_f1"], figsize=(7, 3)); plt.savefig(f"{CFG.WORK}/curves.png", dpi=110)

# confidence threshold: accuracy of the frames we keep vs how many we keep (frames below it are treated as "uncertain")
def sel_table(p, y):
    conf, pr = p.max(1); out = []
    for th in np.arange(0.5, 0.96, 0.05):
        m = conf >= th
        out.append(dict(threshold=round(float(th), 2), coverage=float(m.float().mean()), accuracy_on_kept=float((pr[m] == y[m]).float().mean()) if m.any() else float("nan")))
    return pd.DataFrame(out)
sv, st = sel_table(pv, yv), sel_table(pt, yt)
ok = sv[(sv.accuracy_on_kept >= 0.97) & (sv.coverage >= 0.5)]
CONF_THR = float(ok.threshold.iloc[0]) if len(ok) else 0.8
print("\nconfidence-threshold trade-off on TEST (threshold chosen on VAL = %.2f):" % CONF_THR); print(st.round(3).to_string(index=False))

In [ ]:
# ---- VIDEO-level evaluation: does the *app's own* verdict logic work on videos the model never saw? ----
from gastrovision_video_scoring import video_scoring
vsc = video_scoring(); OTHER = CFG.OTHER_NAME if HAS_OTHER else None
VID_MIN_RUN, VID_SMOOTH = 2, 3          # frames are sampled at 1 fps here (the app uses 3 fps and min_run 3 = the same 1 s)

def eval_videos(dfx, P):
    out = []
    dfx = dfx.reset_index(drop=True)          # FIX: positions must refer to the same frame that P is aligned with
    for v, ix in dfx.groupby("vid").indices.items():
        ix = sorted(ix, key=lambda i: dfx.path.iloc[i])
        s = vsc.summarize_probs([Path(dfx.path.iloc[i]).name for i in ix], CLASSES, P[ix], conf_threshold=CONF_THR,
                                min_run=VID_MIN_RUN, smooth=VID_SMOOTH, other_class=OTHER)
        truth = dfx.cls.iloc[ix[0]]; det = [f["class"] for f in s["findings"]]
        hit = (len(det) == 0) if truth == CFG.OTHER_NAME else (truth in det)
        head_ok = (s["label"] == "no-finding") if truth == CFG.OTHER_NAME else (s["label"] == truth)
        am = Counter(CLASSES[k] for k in P[ix].argmax(1))
        out.append(dict(video=Path(v).stem[:8], truth=truth, n_frames=len(ix), headline=s["label"], detected=det, hit=bool(hit),
                        headline_correct=bool(head_ok), false_findings=len([d for d in det if d != truth]),
                        frame_argmax=dict(am.most_common(3))))
    return out

# frame-level split by domain (test set)
is_vid = D["test"].vid.notna().values; pred_np = pred.numpy(); yt_np = yt.numpy()
for nm, m in (("still images", ~is_vid), ("video frames", is_vid)):
    if m.any(): print(f"TEST frame accuracy on {nm:13s}: {accuracy_score(yt_np[m], pred_np[m]):.4f}  (n={int(m.sum())})")

VIDEO_REPORT = None
tv = eval_videos(D["test"], pt.numpy())
if tv:
    r = pd.DataFrame(tv)
    VIDEO_REPORT = dict(n_videos=len(r), detection_rate=float(r.hit.mean()), headline_accuracy=float(r.headline_correct.mean()),
                        mean_false_findings_per_video=float(r.false_findings.mean()), per_video=tv)
    print(f"\nHELD-OUT VIDEOS ({len(r)}): the labelled finding was detected in {100*r.hit.mean():.0f}% | headline verdict correct in "
          f"{100*r.headline_correct.mean():.0f}% | false extra findings per video {r.false_findings.mean():.2f}")
    print(r.groupby("truth").agg(videos=("hit", "size"), detected=("hit", "sum"), headline_ok=("headline_correct", "sum")).to_string())
    print(r[["video", "truth", "headline", "detected", "frame_argmax"]].to_string(index=False))
else:
    print("no test videos (no videos in the dataset) - video robustness could not be measured")

# external check: the clips you supplied, never used for training
EXT_REPORT = []
if len(EXT):
    pe = F.softmax(predict_logits(model, mk(EXT, train=False)) / TEMP, 1).numpy()
    EXT_REPORT = eval_videos(EXT, pe)
    print("\nEXTERNAL CHECK VIDEOS (excluded from all training):")
    for e in EXT_REPORT: print(f"  {e['video']}  labelled: {e['truth']:22s} -> verdict: {e['headline']:20s} detected: {e['detected']}  frames: {e['n_frames']}  argmax: {e['frame_argmax']}")


### 10 · Export to ONNX with built-in class-activation maps (no PyTorch needed in the app)

In [ ]:
class ExportWrapper(nn.Module):
    """Outputs: logits [B,K] and cam [B,K,h,w].  For a global-average-pool -> linear head,
    CAM_k = sum_c W[k,c] * feature_map_c  and  mean_hw(CAM_k) + b_k == logit_k  (exactly)."""
    def __init__(s, m): super().__init__(); s.m = m; s.fc = m.get_classifier()
    def forward(s, x):
        f = s.m.forward_features(x)
        return s.m.forward_head(f), torch.einsum("kc,bchw->bkhw", s.fc.weight, f)

cpu_model = copy.deepcopy(model).float().cpu().eval()
wrap = ExportWrapper(cpu_model).eval()
dummy = torch.randn(2, 3, IMG, IMG)
with torch.no_grad():
    lg, cam = wrap(dummy)
    identity_err = (cam.mean((2, 3)) + wrap.fc.bias - lg).abs().max().item()
print(f"CAM identity check  max|mean(CAM)+b-logits| = {identity_err:.2e}   (must be ~0)")
assert identity_err < 1e-3, "This backbone's head is not global-pool -> linear; CAM would not be exact. Choose another BACKBONE."

onnx_path = f"{CFG.WORK}/gastrovision_cnn.onnx"
kw = dict(input_names=["pixel_values"], output_names=["logits", "cam"], opset_version=17, do_constant_folding=True,
          dynamic_axes={"pixel_values": {0: "batch"}, "logits": {0: "batch"}, "cam": {0: "batch"}})
try: torch.onnx.export(wrap, dummy, onnx_path, dynamo=False, **kw)
except TypeError: torch.onnx.export(wrap, dummy, onnx_path, **kw)
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
xs = np.stack([gp.pil_to_tensor_array(Image.open(p), IMG, MEAN, STD) for p in D["test"].cache[:8]]).astype(np.float32)
o_logits, o_cam = sess.run(None, {"pixel_values": xs})
with torch.no_grad(): t_logits, t_cam = wrap(torch.from_numpy(xs))
d1, d2 = np.abs(o_logits - t_logits.numpy()).max(), np.abs(o_cam - t_cam.numpy()).max()
print(f"ONNX vs PyTorch  max|d logits| = {d1:.2e}   max|d cam| = {d2:.2e}   | file size {os.path.getsize(onnx_path)/1e6:.1f} MB | cam shape {o_cam.shape}")
assert d1 < 5e-3 and d2 < 5e-2, "ONNX export does not match PyTorch"
# end-to-end check through the *app's* preprocessing on a raw (uncached) image
raw = np.asarray(Image.open(D["test"].path[0]).convert("RGB"))
x_app = gp.preprocess_rgb(raw, dict(img_size=IMG, mean=MEAN, std=STD, cache_max_side=CFG.CACHE_MAX_SIDE))[None]
print("app-preprocessing prediction:", CLASSES[int(sess.run(None, {"pixel_values": x_app})[0].argmax())], "| true:", D["test"].cls[0])

### 11 · Do the heatmaps point at the lesion? (optional — needs Kvasir-SEG attached)

In [ ]:
def find_seg():
    for dp, _, fns in os.walk("/kaggle/input"):
        if os.path.basename(dp).lower() == "masks" and fns:
            im = os.path.join(os.path.dirname(dp), "images")
            if os.path.isdir(im): return im, dp
    return None
cam_report = {"evaluated": 0}
seg = None if CFG.SMOKE else find_seg()
if seg:
    _, mdir = seg
    masks = {Path(f).stem: os.path.join(mdir, f) for f in os.listdir(mdir)}
    hits, ious, base, nev = [], [], [], 0
    pol = D["test"][D["test"].cls == "polyps"]
    cfg_app = dict(img_size=IMG, mean=MEAN, std=STD, cache_max_side=CFG.CACHE_MAX_SIDE)
    for pth in pol.path:
        st_ = Path(pth).stem
        if st_ not in masks: continue
        try:
            rgb = np.asarray(Image.open(pth).convert("RGB")); H, W = rgb.shape[:2]
            m = np.asarray(Image.open(masks[st_]).convert("L").resize((W, H), Image.NEAREST)) > 127
            r0, r1, c0, c1 = gp.black_border_box(rgb); m = m[r0:r1, c0:c1]
            x = torch.from_numpy(gp.preprocess_rgb(rgb, cfg_app))[None]
            with torch.no_grad(): lg_, cm_ = wrap(x)
            k = C2I["polyps"]; cmap = np.maximum(cm_[0, k].numpy(), 0)
            import cv2
            cmap = cv2.resize(cmap, (m.shape[1], m.shape[0]), interpolation=cv2.INTER_CUBIC)
            hits.append(bool(m[np.unravel_index(cmap.argmax(), cmap.shape)]))
            b = cmap >= 0.5 * cmap.max(); ious.append((b & m).sum() / max(1, (b | m).sum())); base.append(m.mean()); nev += 1
        except Exception as e:
            continue
    cam_report = dict(evaluated=nev, pointing_game=float(np.mean(hits)) if nev else None,
                      mean_iou_at_50pct=float(np.mean(ious)) if nev else None, random_baseline_pointing=float(np.mean(base)) if nev else None)
print("heatmap validation:", cam_report if cam_report["evaluated"] else "skipped (attach Kvasir-SEG, SMOKE=False)")

### 12 · Write the model bundle

In [ ]:
classes_json = f"{CFG.WORK}/class_names.json"; json.dump(CLASSES, open(classes_json, "w"), indent=2)
pp = dict(img_size=IMG, mean=MEAN, std=STD, cache_max_side=CFG.CACHE_MAX_SIDE, temperature=TEMP, conf_threshold=CONF_THR,
          other_class=CFG.OTHER_NAME if HAS_OTHER else None, backbone=CFG.BACKBONE, cam_size=list(o_cam.shape[2:]),
          created=datetime.datetime.utcnow().isoformat() + "Z", model_file="gastrovision_cnn.onnx")
json.dump(pp, open(f"{CFG.WORK}/preprocess.json", "w"), indent=2)
metrics = dict(smoke_run=CFG.SMOKE, n_train=len(D["train"]), n_val=len(D["val"]), n_test=len(D["test"]),
               test_accuracy=acc, test_macro_f1=f1m, test_accuracy_tta=acc_tta, test_accuracy_degraded=acc_deg,
               temperature=TEMP, ece_after=ece(pt, yt), conf_threshold=CONF_THR,
               selective_test=st.to_dict("records"), per_class=rep, heatmap_validation=cam_report,
               group_similarity_threshold=thr, video_eval=globals().get("VIDEO_REPORT"), external_videos=globals().get("EXT_REPORT"), has_other_class=HAS_OTHER,
               old_vit_reference="94.5% on an image-level random val split (not leakage-controlled)")
json.dump(metrics, open(f"{CFG.WORK}/metrics.json", "w"), indent=2, default=float)
bundle = Path(CFG.WORK) / "bundle"; shutil.rmtree(bundle, ignore_errors=True); bundle.mkdir()
for f in ["gastrovision_cnn.onnx", "class_names.json", "preprocess.json", "metrics.json", "confusion_matrix.png", "splits.csv", "gastrovision_preprocess.py"]:
    if os.path.exists(f"{CFG.WORK}/{f}"): shutil.copy(f"{CFG.WORK}/{f}", bundle / f)
shutil.make_archive(f"{CFG.WORK}/gastrovision_model_bundle", "zip", bundle)
print("\nBUNDLE READY ->", f"{CFG.WORK}/gastrovision_model_bundle.zip", f"({os.path.getsize(CFG.WORK + '/gastrovision_model_bundle.zip')/1e6:.0f} MB)")
print(f"TEST accuracy {acc:.4f} | macro-F1 {f1m:.4f} | degraded {acc_deg:.4f} | T={TEMP:.2f} | conf threshold {CONF_THR}")
if CFG.SMOKE: print("\nSMOKE RUN PASSED. Now set CFG.SMOKE = False (cell 2) and Run All for the real training.")